## Creating a `Molecule` through `MDMC`, or using a `.cif` file and applying bonds/forcefields to them

In [ ]:
from MDMC.MD import Atom, Molecule, Bond, BondAngle, DihedralAngle, Universe, Dispersion, Shake, PPPM
from MDMC.gui import view
from MDMC.readers.configurations import read

In [ ]:
# Define the unique atoms using the ForceField atom_type
# These can be seen in the oplsaa.dat file (MDMC/MD/force_fields/data/oplsaa.dat)
# The H1 atom will be copied after the bond and bond angles have been defined
HC1 = Atom('H', position=[-0.7006,  0.3636,  0.8900], name='98', charge=0., atom_type=1)
C = Atom('C', position=[-0.3366, -0.1504,  0.0000], name='99', charge=0., atom_type=2)
O = Atom('O', position=[ 1.0849, -0.1713,  0.0000], name='96', charge=0., atom_type=3)
HO = Atom('H', position=[ 1.3606,  0.7699,  0.0000], name='97', charge=0., atom_type=4)
# Create the bonds with harmonic potentials
CH_bond = Bond(C, HC1)
CO_bond = Bond(C, O, constrained=True)
OH_bond = Bond(O, HO)


In [ ]:
# Create the H-C-O and H-O-C bond angles
HCO_angle = BondAngle((HC1, C, O))
HOC_angle = BondAngle((HO, O, C))

# Create the H-C-O-H dihedral
HCOH_dihedral = DihedralAngle((HC1, C, O, HO))

# Duplicate the HC1 atom
HC2 = HC1.copy(position=[-0.7006,  0.3636, -0.8900])

# Create an HCH bond angle
HCH_angle = BondAngle((HC1, C, HC2))

# Duplicate the HC1 atom again
# This atom will have all bond (CH_bond) and bond angles (HCO_angle and HCH_angle) defined
HC3 = HC1.copy(position=[-0.7076, -1.1754,  0.0000])
atoms=[HC1, HC2, HC3, C, O, HO]

# Create the methanol Molecule
methanol = Molecule(atoms=atoms)

In [ ]:
view(atoms)

In [ ]:

# Create a universe and add the methanol
universe = Universe(dimensions=15.0, constraint_algorithm=Shake(1e-5, 100), electrostatic_solver=PPPM(accuracy=1e-4))
universe.fill(methanol, num_density=0.01)

In [ ]:
print(f'There are {universe.n_atoms} atoms in {universe.n_molecules} molecules in this universe')

In [ ]:

# Add dispersion interactions (no dispersion for HO)
HC_disp = Dispersion(universe, (1, 1), cutoff = 8.0, vdw_tail_correction=True)
C_disp = Dispersion(universe, (2, 2), cutoff = 8.0, vdw_tail_correction=True)
O_disp = Dispersion(universe, (3, 3), cutoff = 8.0, vdw_tail_correction=True)
HO_disp = Dispersion(universe, (4, 4), cutoff = 8.0, vdw_tail_correction=True)

In [ ]:
# Add a forcefield to the universe
universe.add_force_field('OPLSAA')

In [ ]:
view(universe)

## Alternatively you can import a .cif file and populate a universe with these molecules

In [ ]:
paracetamol_path = '../../doc/tutorials/data/Paracetamol.cif'
atoms = read(paracetamol_path)
paracetamol = Molecule(atoms=atoms)

In [ ]:
view(paracetamol)

In [ ]:
# To see all of the Bond interactions, we can filter the paracetamol interactions by name
bonds = list(filter(lambda x: x.name == 'Bond', paracetamol.interactions))

# There are 20 bonds in total
print('Number of bonds: {}'.format(len(bonds)))

# We can cast the list of bonds to a set to see the number of unique bonds - in this case it is still 20
unique_bonds = set(bonds)
print('Number of unique bonds: {}'.format(len(unique_bonds)))

### The use of `names`
The `names` must be strings which are only shared by equivalent atoms.  These set the `name` attribute for each `Atom`.  As covered in the tutorial [Applying a Forcefield](applying-a-forcefield.ipynb), if each `Atom` has an `Atom.name` which is defined in a `ForceField`, the `ForceField` can be applied to set the `Parameter` values for all interactions.  

### An example of this is shown below, where the `names` provided are the OPLSAA atom types:

In [ ]:
atoms = read(paracetamol_path, names=['109', '177', # Oxygens
                                      '207', # Nitrogen
                                      '208', '108', '90', '178', '90', '90', '90', '185', # Carbons
                                      '85', '85', '85', '91', '91', '91', '91', '183', '110']) # Hydrogens
paracetamol = Molecule(atoms=atoms)

In [ ]:
universe = Universe(10.)
universe.add_structure(paracetamol)
universe.add_force_field('OPLSAA')

In [ ]:
view(universe)